# Conditional Logit Modelling — Literature-Motivated Main Specification

This notebook reruns the main Conditional Logit experiment using the corrected model-ready dataset with YES/NO trading information and volume/activity features.

## Baseline
- `Lag1_Return`
- `Lag2_Return`
- `Lag3_Return`

## Main Enhanced specification

The Baseline plus five literature-motivated Polymarket information channels:

1. **Belief level**
   - `PM_Prob_Lag1`

2. **Belief innovation / new information**
   - `PM_Change_1D_Lag1`

3. **Horizon dependence**
   - `PM_Prob_x_TimeToResolution`

4. **Cumulative trading activity**
   - `PM_Log_Cumulative_Support_Notional_Lag1`

5. **Directional order flow**
   - `PM_Net_Support_Imbalance_Lag1`

## Why this is the main specification

The corrected dataset contains many related Polymarket variables. The main model deliberately does **not** select features based on final-test performance.

Instead, it uses a compact set representing distinct information channels supported by the literature.

Variables such as:
- cumulative probability change,
- volume surge,
- relative volume,
- participation,
- trade size,
- concentration,
- price dispersion,

remain in the separate RQ2 feature-ablation / robustness analysis.

## Evaluation
- chronological global 80/20 split;
- four expanding-window validation folds inside development;
- Log Loss primary;
- Accuracy and Brier secondary;
- no technical indicators;
- no look-ahead.


## Step 1 — Upload the Corrected Model-Ready Excel File

Upload the corrected dataset:

`Final_Competitive_Markets_MODEL_READY_with_CORRECTED_PM_Trading_Features.xlsx`

The code also accepts a downloaded copy containing `(1)` in the filename.


## Step 2 — Install and Import Required Packages


In [ ]:
# ============================================================
# STEP 2: Install and Import Packages
# ============================================================

!pip -q install statsmodels openpyxl scipy scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.special import softmax
from scipy.stats import chi2

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

from statsmodels.discrete.conditional_models import ConditionalLogit
import statsmodels.api as sm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Packages imported successfully.")


## Step 3 — Load the Model-Ready Master Sheet


In [ ]:
# ============================================================
# STEP 3: Load Corrected Model-Ready Data
# ============================================================

import glob
import os

candidates = glob.glob(
    "/content/*CORRECTED_PM_Trading_Features*.xlsx"
)

if len(candidates) == 0:
    raise FileNotFoundError(
        "Upload the corrected model-ready workbook containing "
        "'CORRECTED_PM_Trading_Features' in its filename."
    )

# Prefer the dataset file, not any results workbook.
dataset_candidates = [
    x for x in candidates
    if "MODEL_READY" in os.path.basename(x).upper()
]

if len(dataset_candidates) != 1:
    raise ValueError(
        "Could not uniquely identify the corrected model-ready dataset. "
        f"Candidates: {dataset_candidates}"
    )

FILE_PATH = dataset_candidates[0]

df = pd.read_excel(
    FILE_PATH,
    sheet_name="Model_Ready_Master"
)

date_columns = [
    "Date",
    "Next_Financial_Date",
    "Financial_Prev1_Date",
    "Financial_Prev2_Date",
    "Financial_Prev3_Date",
    "PM_Signal_Date_Lag1",
    "PM_Refined_Activity_Signal_Date_Lag1"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

df = df.sort_values(
    ["Date", "Market_ID", "Event_ID", "Asset"]
).reset_index(drop=True)

print("Loaded:", FILE_PATH)
print("Rows:", len(df))
print("Event IDs:", df["Event_ID"].nunique())
print("Unique dates:", df["Date"].nunique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())

print("\nEvents by market:")
print(df.groupby("Market_ID")["Event_ID"].nunique())

display(df.head())


## Step 4 — Define the Baseline and New Enhanced Feature Sets

The new Enhanced model combines the original probability variables with a **compact corrected trading-activity core**.

`Days_to_Resolution_Years` is still not entered alone in Conditional Logit because it is constant across alternatives inside an Event ID.


In [ ]:
BASELINE_FEATURES = [
    "Lag1_Return",
    "Lag2_Return",
    "Lag3_Return"
]

# Literature-motivated main Polymarket information channels.
PM_MAIN_FEATURES = [
    "PM_Prob_Lag1",
    "PM_Change_1D_Lag1",
    "PM_Prob_x_TimeToResolution",
    "PM_Log_Cumulative_Support_Notional_Lag1",
    "PM_Net_Support_Imbalance_Lag1"
]

ENHANCED_FEATURES = (
    BASELINE_FEATURES
    + PM_MAIN_FEATURES
)

# These are valid robustness / RQ2 features, but are intentionally
# NOT part of the headline Enhanced specification.
RQ2_ONLY_FEATURES = [
    "PM_Change_From_Start_Lag1",
    "PM_Log_Support_Notional_Lag1",
    "PM_Relative_Support_Volume_Share_Lag1",
    "PM_Relative_Cumulative_Support_Share_Lag1",
    "PM_Log_Support_Unique_Takers_Lag1",
    "PM_Support_Volume_Surge_Lag1",
    "PM_Prob_x_Log_Contract_Notional_Lag1",
    "PM_Support_HHI_Lag1",
    "PM_Log_Support_Mean_Trade_Notional_Lag1",
    "PM_Candidate_Price_Dispersion_Lag1"
]

NEVER_USE_AS_PREDICTORS = [
    "Winner",
    "Forward_Log_Return",
    "Price_Next",
    "PM_Prob_Current_Audit"
]

feature_set_table = pd.DataFrame([
    {
        "Specification": "Baseline",
        "Model_Family": "Conditional Logit",
        "Features": " + ".join(BASELINE_FEATURES)
    },
    {
        "Specification": "Enhanced",
        "Model_Family": "Conditional Logit",
        "Features": " + ".join(ENHANCED_FEATURES)
    }
])

print("BASELINE FEATURES")
print(BASELINE_FEATURES)

print("\nLITERATURE-MOTIVATED MAIN POLYMARKET FEATURES")
print(PM_MAIN_FEATURES)

print("\nMAIN ENHANCED FEATURES")
print(ENHANCED_FEATURES)

print("\nRQ2 / ROBUSTNESS-ONLY FEATURES")
print(RQ2_ONLY_FEATURES)


## Step 5 — Final Leakage and Structure Check

This is performed again immediately before modelling.


In [ ]:
# ============================================================
# STEP 5: Leakage and Structural Checks
# ============================================================

# Exactly one winner per event
winner_counts = df.groupby("Event_ID")["Winner"].sum()

assert (winner_counts == 1).all(),     "ERROR: Some Event IDs do not contain exactly one winner."

# Event rows intact
choice_sizes = df.groupby("Event_ID")["Asset"].nunique()
assert choice_sizes.isin([2, 3]).all(),     "ERROR: Unexpected choice-set size."

# Polymarket signal strictly before forecast date
assert (
    df["PM_Signal_Date_Lag1"] < df["Date"]
).all(), "ERROR: Polymarket timing leakage detected."

# Corrected Polymarket trading signal strictly before forecast date
assert (
    df["PM_Refined_Activity_Signal_Date_Lag1"] < df["Date"]
).all(), "ERROR: Corrected Polymarket trading-activity timing leakage detected."

# Previous financial dates strictly before forecast date
assert (
    df["Financial_Prev1_Date"] < df["Date"]
).all(), "ERROR: Financial lag timing problem detected."

# No missing modelling features
all_required = list(dict.fromkeys(
    BASELINE_FEATURES + ENHANCED_FEATURES
))

assert df[all_required].notna().all().all(),     "ERROR: Missing predictor values detected."

print("All leakage and structural checks PASSED.")

print("\nChoice-set sizes:")
print(choice_sizes.value_counts().sort_index())

print("\nWinner count per event:")
print(winner_counts.value_counts().sort_index())


## Step 6 — Freeze the Chronological Development / Final-Test Split

The split is based on **unique forecast dates**, not rows.

Approximately:
- First 80% of unique dates → Development
- Last 20% of unique dates → Final untouched test

If multiple markets occur on the same calendar date, they remain together.


In [ ]:
# ============================================================
# STEP 6: Freeze Development and Final Test Dates
# ============================================================

unique_dates = np.array(
    sorted(df["Date"].dropna().unique())
)

n_dates = len(unique_dates)
n_dev_dates = int(np.floor(0.80 * n_dates))

dev_dates = unique_dates[:n_dev_dates]
test_dates = unique_dates[n_dev_dates:]

dev = df[df["Date"].isin(dev_dates)].copy()
final_test = df[df["Date"].isin(test_dates)].copy()

print("Total unique dates:", n_dates)
print("Development dates:", len(dev_dates))
print("Final test dates:", len(test_dates))

print("\nDEVELOPMENT")
print("Date range:", dev["Date"].min(), "to", dev["Date"].max())
print("Events:", dev["Event_ID"].nunique())
print("Rows:", len(dev))

print("\nFINAL TEST — KEEP UNTOUCHED UNTIL FINAL EVALUATION")
print("Date range:", final_test["Date"].min(), "to", final_test["Date"].max())
print("Events:", final_test["Event_ID"].nunique())
print("Rows:", len(final_test))

print("\nFinal-test events by market:")
print(
    final_test.groupby("Market_ID")["Event_ID"].nunique()
)


## Step 7 — Create Four Expanding-Window Validation Folds

The four folds are created only inside the development period.

Each validation block contains later dates than its corresponding training block.


In [ ]:
# ============================================================
# STEP 7: Create Expanding-Window Validation Folds
# ============================================================

# Four validation blocks of 25 unique forecast dates each.
# With 218 development dates, this leaves 118 dates
# for the first training window.
tscv = TimeSeriesSplit(
    n_splits=4,
    test_size=25,
    gap=0
)

cv_folds = []

for fold_no, (train_date_idx, val_date_idx) in enumerate(
    tscv.split(dev_dates),
    start=1
):
    train_dates_fold = dev_dates[train_date_idx]
    val_dates_fold = dev_dates[val_date_idx]

    train_fold = df[
        df["Date"].isin(train_dates_fold)
    ].copy()

    val_fold = df[
        df["Date"].isin(val_dates_fold)
    ].copy()

    # Strict chronological order
    assert train_fold["Date"].max() < val_fold["Date"].min()

    # No event overlap
    assert set(train_fold["Event_ID"]).isdisjoint(
        set(val_fold["Event_ID"])
    )

    # No date overlap
    assert set(train_fold["Date"]).isdisjoint(
        set(val_fold["Date"])
    )

    cv_folds.append({
        "Fold": fold_no,
        "Train_Dates": train_dates_fold,
        "Val_Dates": val_dates_fold
    })

    print(f"\nFOLD {fold_no}")
    print(
        "Train:",
        train_fold["Date"].min().date(),
        "to",
        train_fold["Date"].max().date(),
        "| Events:",
        train_fold["Event_ID"].nunique()
    )
    print(
        "Validation:",
        val_fold["Date"].min().date(),
        "to",
        val_fold["Date"].max().date(),
        "| Events:",
        val_fold["Event_ID"].nunique()
    )


## Step 8 — Helper Functions for Conditional Logit and Event Probabilities

Features are standardized using the **training sample only**.

Predicted utilities are converted to probabilities separately within each Event ID using a softmax transformation, so predicted probabilities sum to 1 within each competition.


In [ ]:
# ============================================================
# STEP 8A: Fit Conditional Logit and Predict Event Probabilities
# ============================================================

def fit_clogit_and_predict(train_df, test_df, features):
    # Fit scaler on training data only
    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train_df[features]
    )

    X_test = scaler.transform(
        test_df[features]
    )

    y_train = train_df["Winner"].astype(int).to_numpy()
    groups_train = train_df["Event_ID"].astype(str).to_numpy()

    model = ConditionalLogit(
        y_train,
        X_train,
        groups=groups_train
    )

    result = model.fit(
        method="bfgs",
        maxiter=1000,
        disp=False
    )

    # Utility scores for test alternatives
    utility = X_test @ np.asarray(result.params)

    pred = test_df[
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "Asset",
            "Winner"
        ]
    ].copy()

    pred["Utility"] = utility
    pred["Pred_Prob"] = np.nan

    # Softmax within each Event ID
    for event_id, idx in pred.groupby("Event_ID").groups.items():
        idx = list(idx)
        pred.loc[idx, "Pred_Prob"] = softmax(
            pred.loc[idx, "Utility"].to_numpy()
        )

    return result, scaler, pred


In [ ]:
# ============================================================
# STEP 8B: Event-Level Evaluation Metrics
# ============================================================

def evaluate_event_predictions(pred):
    eps = 1e-15

    event_rows = []

    for event_id, g in pred.groupby("Event_ID"):
        g = g.copy()

        # Ensure probabilities sum to one
        probs = g["Pred_Prob"].to_numpy(dtype=float)
        probs = probs / probs.sum()

        y = g["Winner"].to_numpy(dtype=int)

        actual_idx = np.where(y == 1)[0][0]
        predicted_idx = int(np.argmax(probs))

        actual_winner = g.iloc[actual_idx]["Asset"]
        predicted_winner = g.iloc[predicted_idx]["Asset"]

        winner_prob = float(
            np.clip(probs[actual_idx], eps, 1 - eps)
        )

        log_loss_event = -np.log(winner_prob)

        # Multiclass Brier score
        brier_event = np.sum(
            (probs - y) ** 2
        )

        correct = int(
            actual_winner == predicted_winner
        )

        J = len(g)

        event_rows.append({
            "Market_ID": g["Market_ID"].iloc[0],
            "Event_ID": event_id,
            "Date": g["Date"].iloc[0],
            "Choice_Set_Size": J,
            "Actual_Winner": actual_winner,
            "Predicted_Winner": predicted_winner,
            "Winner_Probability": winner_prob,
            "Correct": correct,
            "Log_Loss": log_loss_event,
            "Brier_Score": brier_event,
            "Uniform_Log_Loss": np.log(J),
            "Uniform_Brier": 1 - (1 / J)
        })

    event_metrics = pd.DataFrame(event_rows)

    mean_log_loss = event_metrics["Log_Loss"].mean()
    mean_brier = event_metrics["Brier_Score"].mean()
    accuracy = event_metrics["Correct"].mean()

    uniform_log = event_metrics["Uniform_Log_Loss"].mean()
    uniform_brier = event_metrics["Uniform_Brier"].mean()

    log_skill = 1 - (
        event_metrics["Log_Loss"].sum() /
        event_metrics["Uniform_Log_Loss"].sum()
    )

    brier_skill = 1 - (
        event_metrics["Brier_Score"].sum() /
        event_metrics["Uniform_Brier"].sum()
    )

    chance_accuracy = (
        (1 / event_metrics["Choice_Set_Size"]).mean()
    )

    summary = {
        "Events": len(event_metrics),
        "Log_Loss": mean_log_loss,
        "Brier_Score": mean_brier,
        "Accuracy": accuracy,
        "Chance_Accuracy": chance_accuracy,
        "Log_Skill_vs_Uniform": log_skill,
        "Brier_Skill_vs_Uniform": brier_skill
    }

    return summary, event_metrics


## Step 9 — Run Expanding-Window Validation: Baseline vs Enhanced

The same dates are used for both models in every fold.


In [ ]:
# ============================================================
# STEP 9: Expanding-Window Cross-Validation
# ============================================================

cv_results = []
cv_event_predictions = []

for fold_info in cv_folds:

    fold_no = fold_info["Fold"]

    train_fold = df[
        df["Date"].isin(
            fold_info["Train_Dates"]
        )
    ].copy()

    val_fold = df[
        df["Date"].isin(
            fold_info["Val_Dates"]
        )
    ].copy()

    print("=" * 75)
    print(f"RUNNING FOLD {fold_no}")
    print("=" * 75)

    for model_name, features in [
        ("Baseline", BASELINE_FEATURES),
        ("Enhanced", ENHANCED_FEATURES)
    ]:

        result, scaler, pred = fit_clogit_and_predict(
            train_fold,
            val_fold,
            features
        )

        metrics, event_metrics = (
            evaluate_event_predictions(pred)
        )

        metrics["Fold"] = fold_no
        metrics["Model"] = model_name

        cv_results.append(metrics)

        event_metrics["Fold"] = fold_no
        event_metrics["Model"] = model_name
        cv_event_predictions.append(event_metrics)

        print(
            f"{model_name}: "
            f"Log Loss={metrics['Log_Loss']:.4f} | "
            f"Brier={metrics['Brier_Score']:.4f} | "
            f"Accuracy={metrics['Accuracy']:.3f}"
        )

cv_results_df = pd.DataFrame(cv_results)
cv_event_predictions_df = pd.concat(
    cv_event_predictions,
    ignore_index=True
)

display(
    cv_results_df.sort_values(
        ["Fold", "Model"]
    )
)


## Step 10 — Summarize Validation Performance Across Folds

For Log Loss and Brier Score: **lower is better**.

For Accuracy and skill scores: **higher is better**.


In [ ]:
# ============================================================
# STEP 10: Cross-Validation Summary
# ============================================================

cv_summary = (
    cv_results_df
    .groupby("Model")
    .agg(
        Mean_Log_Loss=("Log_Loss", "mean"),
        SD_Log_Loss=("Log_Loss", "std"),
        Mean_Brier=("Brier_Score", "mean"),
        SD_Brier=("Brier_Score", "std"),
        Mean_Accuracy=("Accuracy", "mean"),
        SD_Accuracy=("Accuracy", "std"),
        Mean_Log_Skill=("Log_Skill_vs_Uniform", "mean"),
        Mean_Brier_Skill=("Brier_Skill_vs_Uniform", "mean")
    )
    .reset_index()
)

display(cv_summary)

# Direct Enhanced - Baseline differences fold by fold
wide_cv = cv_results_df.pivot(
    index="Fold",
    columns="Model",
    values=[
        "Log_Loss",
        "Brier_Score",
        "Accuracy"
    ]
)

cv_differences = pd.DataFrame({
    "Fold": wide_cv.index,
    "LogLoss_Enhanced_minus_Baseline":
        wide_cv["Log_Loss"]["Enhanced"] -
        wide_cv["Log_Loss"]["Baseline"],
    "Brier_Enhanced_minus_Baseline":
        wide_cv["Brier_Score"]["Enhanced"] -
        wide_cv["Brier_Score"]["Baseline"],
    "Accuracy_Enhanced_minus_Baseline":
        wide_cv["Accuracy"]["Enhanced"] -
        wide_cv["Accuracy"]["Baseline"]
}).reset_index(drop=True)

print(
    "\nInterpretation:"
    "\nNegative LogLoss difference = Enhanced is better."
    "\nNegative Brier difference = Enhanced is better."
    "\nPositive Accuracy difference = Enhanced is better."
)

display(cv_differences)


## Step 11 — Fit Baseline and Enhanced Conditional Logit on the Full Development Sample

These are the final model specifications before touching the final holdout.


In [ ]:
# ============================================================
# STEP 11: Fit Final Development Models
# ============================================================

baseline_dev_result, baseline_scaler, _ = (
    fit_clogit_and_predict(
        dev,
        dev,
        BASELINE_FEATURES
    )
)

enhanced_dev_result, enhanced_scaler, _ = (
    fit_clogit_and_predict(
        dev,
        dev,
        ENHANCED_FEATURES
    )
)

print("Baseline development log-likelihood:",
      baseline_dev_result.llf)

print("Enhanced development log-likelihood:",
      enhanced_dev_result.llf)


## Step 12 — Development-Sample Coefficients and Likelihood-Ratio Test

This is a **secondary in-sample diagnostic**.

The main dissertation conclusion must come from out-of-sample validation and final-test performance.


In [ ]:
# ============================================================
# STEP 12A: Coefficient Tables
# ============================================================

def coefficient_table(result, features, model_name):
    params = np.asarray(result.params)
    bse = np.asarray(result.bse)
    pvalues = np.asarray(result.pvalues)

    return pd.DataFrame({
        "Model": model_name,
        "Feature": features,
        "Coefficient_Standardized": params,
        "Std_Error": bse,
        "P_Value": pvalues
    })

baseline_coef = coefficient_table(
    baseline_dev_result,
    BASELINE_FEATURES,
    "Baseline"
)

enhanced_coef = coefficient_table(
    enhanced_dev_result,
    ENHANCED_FEATURES,
    "Enhanced"
)

development_coefficients = pd.concat(
    [baseline_coef, enhanced_coef],
    ignore_index=True
)

display(development_coefficients)


In [ ]:
# ============================================================
# STEP 12B: Likelihood-Ratio Test
# ============================================================

ll_base = baseline_dev_result.llf
ll_enh = enhanced_dev_result.llf

lr_stat = 2 * (ll_enh - ll_base)
df_diff = (
    len(ENHANCED_FEATURES) -
    len(BASELINE_FEATURES)
)

lr_pvalue = chi2.sf(
    lr_stat,
    df=df_diff
)

n_dev_events = dev["Event_ID"].nunique()

aic_base = (
    2 * len(BASELINE_FEATURES) -
    2 * ll_base
)

aic_enh = (
    2 * len(ENHANCED_FEATURES) -
    2 * ll_enh
)

bic_base = (
    len(BASELINE_FEATURES) *
    np.log(n_dev_events) -
    2 * ll_base
)

bic_enh = (
    len(ENHANCED_FEATURES) *
    np.log(n_dev_events) -
    2 * ll_enh
)

lr_test_table = pd.DataFrame({
    "Statistic": [
        "Baseline Log-Likelihood",
        "Enhanced Log-Likelihood",
        "Likelihood-Ratio Statistic",
        "Degrees of Freedom",
        "LR p-value",
        "Baseline AIC",
        "Enhanced AIC",
        "Baseline BIC",
        "Enhanced BIC"
    ],
    "Value": [
        ll_base,
        ll_enh,
        lr_stat,
        df_diff,
        lr_pvalue,
        aic_base,
        aic_enh,
        bic_base,
        bic_enh
    ]
})

display(lr_test_table)


## Step 13 — FINAL UNTOUCHED TEST: Baseline vs Enhanced

This is the key out-of-sample evaluation.

The models are trained using the complete development period and then evaluated once on the latest ~20% of forecast dates.


In [ ]:
# ============================================================
# STEP 13: Final Out-of-Sample Test
# ============================================================

final_results = []
final_event_predictions = []
final_row_predictions = []

for model_name, features in [
    ("Baseline", BASELINE_FEATURES),
    ("Enhanced", ENHANCED_FEATURES)
]:

    result, scaler, pred = fit_clogit_and_predict(
        dev,
        final_test,
        features
    )

    metrics, event_metrics = (
        evaluate_event_predictions(pred)
    )

    metrics["Model"] = model_name
    final_results.append(metrics)

    event_metrics["Model"] = model_name
    final_event_predictions.append(event_metrics)

    pred["Model"] = model_name
    final_row_predictions.append(pred)

final_results_df = pd.DataFrame(
    final_results
)

final_event_predictions_df = pd.concat(
    final_event_predictions,
    ignore_index=True
)

final_row_predictions_df = pd.concat(
    final_row_predictions,
    ignore_index=True
)

display(final_results_df)

print(
    "\nRemember:"
    "\nLower Log Loss = better."
    "\nLower Brier Score = better."
    "\nHigher Accuracy = better."
)


## Step 14 — Calculate the Final Polymarket Improvement

This directly answers the main research question.

For loss metrics, a negative Enhanced − Baseline difference means improvement.

For Accuracy, a positive difference means improvement.


In [ ]:
# ============================================================
# STEP 14: Final Enhanced vs Baseline Differences
# ============================================================

base_row = final_results_df[
    final_results_df["Model"] == "Baseline"
].iloc[0]

enh_row = final_results_df[
    final_results_df["Model"] == "Enhanced"
].iloc[0]

final_difference = pd.DataFrame({
    "Metric": [
        "Log Loss",
        "Brier Score",
        "Accuracy"
    ],
    "Baseline": [
        base_row["Log_Loss"],
        base_row["Brier_Score"],
        base_row["Accuracy"]
    ],
    "Enhanced": [
        enh_row["Log_Loss"],
        enh_row["Brier_Score"],
        enh_row["Accuracy"]
    ]
})

final_difference["Enhanced_minus_Baseline"] = (
    final_difference["Enhanced"] -
    final_difference["Baseline"]
)

display(final_difference)


## Step 15 — Final Performance by Market

Because the study includes both 2-way and 3-way competitions, overall results should be supplemented with market-specific results.


In [ ]:
# ============================================================
# STEP 15: Final Performance by Market
# ============================================================

market_results = []

for model_name in ["Baseline", "Enhanced"]:

    model_events = final_event_predictions_df[
        final_event_predictions_df["Model"] == model_name
    ]

    for market_id, g in model_events.groupby("Market_ID"):

        market_results.append({
            "Model": model_name,
            "Market_ID": market_id,
            "Events": len(g),
            "Log_Loss": g["Log_Loss"].mean(),
            "Brier_Score": g["Brier_Score"].mean(),
            "Accuracy": g["Correct"].mean(),
            "Chance_Accuracy": (
                1 / g["Choice_Set_Size"]
            ).mean()
        })

final_by_market = pd.DataFrame(
    market_results
)

display(
    final_by_market.sort_values(
        ["Market_ID", "Model"]
    )
)


## Step 16 — Date-Level Loss Comparison

Multiple competitive markets can occur on the same date and share underlying financial outcomes.

Therefore baseline-versus-enhanced loss comparisons are aggregated to the **Date level** before statistical comparison.


In [ ]:
# ============================================================
# STEP 16A: Build Date-Level Loss Differences
# ============================================================

date_model_losses = (
    final_event_predictions_df
    .groupby(["Date", "Model"])
    .agg(
        Mean_Log_Loss=("Log_Loss", "mean"),
        Mean_Brier=("Brier_Score", "mean"),
        Mean_Accuracy=("Correct", "mean")
    )
    .reset_index()
)

date_wide = date_model_losses.pivot(
    index="Date",
    columns="Model",
    values=[
        "Mean_Log_Loss",
        "Mean_Brier",
        "Mean_Accuracy"
    ]
)

date_loss_differences = pd.DataFrame({
    "Date": date_wide.index,
    "LogLoss_Baseline_minus_Enhanced":
        date_wide["Mean_Log_Loss"]["Baseline"] -
        date_wide["Mean_Log_Loss"]["Enhanced"],
    "Brier_Baseline_minus_Enhanced":
        date_wide["Mean_Brier"]["Baseline"] -
        date_wide["Mean_Brier"]["Enhanced"],
    "Accuracy_Enhanced_minus_Baseline":
        date_wide["Mean_Accuracy"]["Enhanced"] -
        date_wide["Mean_Accuracy"]["Baseline"]
}).reset_index(drop=True)

print(
    "Positive loss difference means Enhanced has lower loss."
)

display(date_loss_differences.head())


## Step 17 — HAC / Diebold-Mariano-Style Loss-Difference Test

This tests whether the mean date-level loss difference differs from zero while allowing serial dependence through a Newey-West HAC covariance estimate.

This is supporting evidence, not the only basis for the dissertation conclusion.


In [ ]:
# ============================================================
# STEP 17: HAC Loss-Difference Tests
# ============================================================

def hac_mean_test(series, maxlags=5):
    x = pd.Series(series).dropna().astype(float)

    X = np.ones((len(x), 1))

    model = sm.OLS(
        x.to_numpy(),
        X
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": maxlags}
    )

    return {
        "Mean_Difference": model.params[0],
        "HAC_Std_Error": model.bse[0],
        "t_Statistic": model.tvalues[0],
        "p_Value": model.pvalues[0],
        "N_Dates": len(x),
        "HAC_MaxLags": maxlags
    }

log_test = hac_mean_test(
    date_loss_differences[
        "LogLoss_Baseline_minus_Enhanced"
    ],
    maxlags=5
)

brier_test = hac_mean_test(
    date_loss_differences[
        "Brier_Baseline_minus_Enhanced"
    ],
    maxlags=5
)

hac_tests = pd.DataFrame([
    {
        "Metric": "Log Loss",
        **log_test
    },
    {
        "Metric": "Brier Score",
        **brier_test
    }
])

display(hac_tests)

print(
    "\nFor the loss differences:"
    "\nPositive mean difference = Enhanced is better."
)


## Step 18 — Create a Frozen Split Manifest

This records exactly which dates and events were used for development and final testing.


In [ ]:
# ============================================================
# STEP 18: Save Split Manifest
# ============================================================

manifest_rows = []

for event_id, g in df.groupby("Event_ID"):

    event_date = g["Date"].iloc[0]

    split = (
        "Development"
        if event_date in set(dev_dates)
        else "Final_Test"
    )

    manifest_rows.append({
        "Event_ID": event_id,
        "Date": event_date,
        "Market_ID": g["Market_ID"].iloc[0],
        "Choice_Set_Size": len(g),
        "Final_Split": split
    })

split_manifest = pd.DataFrame(
    manifest_rows
).sort_values(
    ["Date", "Market_ID", "Event_ID"]
)

display(split_manifest.head())


## Step 19 — Save All Conditional Logit Results to Excel

The workbook contains:

- CV fold results
- CV summary
- CV baseline/enhanced differences
- development coefficient tables
- likelihood-ratio/AIC/BIC diagnostics
- final-test overall results
- final-test market-specific results
- final baseline/enhanced differences
- final event-level predictions
- final alternative-level probabilities
- date-level loss differences
- HAC comparison tests
- split manifest


In [ ]:
# ============================================================
# STEP 19: Save Results to Excel
# ============================================================

OUTPUT_FILE = (
    "/content/"
    "Conditional_Logit_Final_Evaluation_All_Results.xlsx"
)

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    cv_results_df.to_excel(
        writer,
        sheet_name="CV_Fold_Results",
        index=False
    )

    cv_summary.to_excel(
        writer,
        sheet_name="CV_Summary",
        index=False
    )

    cv_differences.to_excel(
        writer,
        sheet_name="CV_Differences",
        index=False
    )

    cv_event_predictions_df.to_excel(
        writer,
        sheet_name="CV_Event_Predictions",
        index=False
    )

    development_coefficients.to_excel(
        writer,
        sheet_name="Development_Coefficients",
        index=False
    )

    lr_test_table.to_excel(
        writer,
        sheet_name="LR_AIC_BIC",
        index=False
    )

    final_results_df.to_excel(
        writer,
        sheet_name="Final_Test_Overall",
        index=False
    )

    final_by_market.to_excel(
        writer,
        sheet_name="Final_Test_By_Market",
        index=False
    )

    final_difference.to_excel(
        writer,
        sheet_name="Final_Differences",
        index=False
    )

    final_event_predictions_df.to_excel(
        writer,
        sheet_name="Final_Event_Predictions",
        index=False
    )

    final_row_predictions_df.to_excel(
        writer,
        sheet_name="Final_Row_Probabilities",
        index=False
    )

    date_loss_differences.to_excel(
        writer,
        sheet_name="Date_Loss_Differences",
        index=False
    )

    hac_tests.to_excel(
        writer,
        sheet_name="HAC_Loss_Tests",
        index=False
    )

    split_manifest.to_excel(
        writer,
        sheet_name="Split_Manifest",
        index=False
    )

    feature_set_table.to_excel(
        writer,
        sheet_name="Feature_Set_Definitions",
        index=False
    )

print("Saved successfully:")
print(OUTPUT_FILE)


## Step 20 — Download the Conditional Logit Results


In [ ]:
# ============================================================
# STEP 20: Download Results Workbook
# ============================================================

from google.colab import files

files.download(
    "/content/"
    "Conditional_Logit_Final_Evaluation_All_Results.xlsx"
)
